In [70]:
import pandas as pd
import numpy as np

In [71]:
df = pd.read_parquet("../data/processed/pdb_protein_features.parquet")

In [72]:
df.columns

Index(['pdb_id', 'method', 'resolution', 'wilson_b', 'organism', 'taxonomy_id',
       'sequence', 'coverage', 'bfactor'],
      dtype='str')

In [73]:
df = df[df["method"] == "X-RAY DIFFRACTION"]

In [74]:
df.shape

(291282, 9)

In [75]:
df = df[(df["wilson_b"].notnull()) & (df["wilson_b"] > 0)]

In [76]:
df.shape

(129812, 9)

In [77]:
def normalize_bfactors(df):
    df = df.copy()
    denom = df["wilson_b"]
    df["b_norm"] = df["bfactor"] / denom
    return df

In [78]:
df = normalize_bfactors(df)

In [80]:
# Применяем фильтры:
# 1. NaN в bfactor меньше 10%
# 2. Длина цепочки больше 100
# 3. Средний bfactor нормированный меньше 20
df["bnorm_mean"] = df["b_norm"].apply(np.nanmean)
# Рассчитываем процент NaN в bfactor для каждой цепи
nan_percent = df.groupby("pdb_id")["bfactor"].apply(
    lambda x: (x.isna().sum() / len(x)) * 100
)
valid_pdbs = nan_percent[nan_percent < 10].index

# Добавляем длину цепочки из sequence
df["chain_length"] = df["sequence"].str.len()

# Фильтруем данные
filtered_df = df[
    (df["pdb_id"].isin(valid_pdbs))  # NaN в bfactor меньше 10%
    & (df["chain_length"] > 100)  # Длина цепочки больше 100
    & (df["bnorm_mean"] < 20)  # Средний bfactor нормированный меньше 20
]

print(f"Исходное количество строк: {len(df)}")
print(f"Фильтровано строк: {len(filtered_df)}")
print(
    f"Удалено: {len(df) - len(filtered_df)} ({((len(df) - len(filtered_df)) / len(df) * 100):.2f}%)"
)

Исходное количество строк: 129812
Фильтровано строк: 108881
Удалено: 20931 (16.12%)


In [81]:
df = filtered_df

In [82]:
duplicates = df[df.duplicated(subset=["sequence", "taxonomy_id"], keep=False)]

In [83]:
duplicates.shape

(66959, 12)

In [84]:
duplicates.head()

,pdb_id,method,resolution,wilson_b,organism,taxonomy_id,sequence,coverage,bfactor,b_norm,chain_length,bnorm_mean
4,102m,X-RAY DIFFRACTION,1.84,12.3,Physeter catodon,9755,MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDR...,1111111111111111111111111111111111111111111111...,"[32.9, 27.92, 20.97, 17.7, 13.15, 13.72, 9.54,...","[2.6747968, 2.2699187, 1.704878, 1.4390244, 1....",154,1.162211
6,103m,X-RAY DIFFRACTION,2.07,18.6,Physeter catodon,9755,MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDR...,1111111111111111111111111111111111111111111111...,"[41.04, 35.53, 28.69, 25.48, 26.72, 21.36, 16....","[2.2064517, 1.9102149, 1.5424731, 1.3698925, 1...",154,1.282021
8,104m,X-RAY DIFFRACTION,1.71,15.2,Physeter catodon,9755,VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRF...,1111111111111111111111111111111111111111111111...,"[20.5, 15.95, 13.9, 12.39, 13.69, 10.71, 9.55,...","[1.3486842, 1.0493422, 0.91447365, 0.8151316, ...",153,0.774901
9,105m,X-RAY DIFFRACTION,2.02,12.9,Physeter catodon,9755,VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRF...,1111111111111111111111111111111111111111111111...,"[30.0, 28.95, 29.11, 24.51, 20.58, 17.75, 16.9...","[2.3255816, 2.2441862, 2.2565892, 1.9000001, 1...",153,1.663495
11,106m,X-RAY DIFFRACTION,1.99,11.5,Physeter catodon,9755,MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDR...,1111111111111111111111111111111111111111111111...,"[32.86, 27.2, 20.55, 18.05, 19.76, 15.24, 11.0...","[2.8573914, 2.3652174, 1.7869564, 1.5695652, 1...",154,1.412722


In [85]:
df = df.sort_values(by=["resolution"], ascending=[True]).drop_duplicates(
    subset=["sequence", "taxonomy_id"], keep="first"
)

In [86]:
df.head()

,pdb_id,method,resolution,wilson_b,organism,taxonomy_id,sequence,coverage,bfactor,b_norm,chain_length,bnorm_mean
375134,6s2m,X-RAY DIFFRACTION,0.72,9.000,Homo sapiens,9606,GMSNKFLGTWKLVSSENFDDYMKALGVGLATRKLGNLAKPTVIISK...,1111111111111111111111111111111111111111111111...,"[25.25, 14.04, 11.68, 9.94, 9.32, 8.01, 8.01, ...","[2.8055556, 1.56, 1.2977778, 1.1044444, 1.0355...",133,0.966291
46752,8c5n,X-RAY DIFFRACTION,0.75,6.900,Pseudomonas aeruginosa PA14,652611,HGSMETPPSRVYGCFLEGPENPKSAACKAAVAAGGTQALYDWNGVN...,1111111111111111111111111111111111111111111111...,"[6.75, 6.26, 5.58, 5.94, 6.3, 6.59, 7.2, 6.74,...","[0.9782609, 0.9072464, 0.8086956, 0.8608696, 0...",193,1.180185
299786,2ov0,X-RAY DIFFRACTION,0.75,4.000,Paracoccus denitrificans,266,DKATIPSESPFAAAEVADGAIVVDIAKMKYETPELHVKVGDTVTWI...,1111111111111111111111111111111111111111111111...,"[12.38, 7.23, 7.56, 7.49, 5.76, 7.96, 5.56, 6....","[3.095, 1.8075, 1.89, 1.8725, 1.44, 1.99, 1.39...",105,1.523190
220245,7kr0,X-RAY DIFFRACTION,0.77,8.481,Severe acute respiratory syndrome coronavirus 2,2697049,SNAGEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYL...,0000111111111111111111111111111111111111111111...,"[nan, nan, nan, nan, 24.04, 15.98, 8.12, 8.86,...","[nan, nan, nan, nan, 2.8345716, 1.8842118, 0.9...",173,0.759749
486933,1x6z,X-RAY DIFFRACTION,0.78,6.800,Pseudomonas aeruginosa,287,ALEGTEFARSEGASALASVNPLKTTVEEALSRGWSVKSGTGTEDAT...,0001111111111111111111111111111111111111111111...,"[nan, nan, nan, 21.16, 12.6, 12.49, 10.39, 10....","[nan, nan, nan, 3.1117647, 1.8529412, 1.836764...",123,1.573909


In [87]:
df.shape

(55800, 12)

In [88]:
duplicates = df[df.duplicated(subset=["sequence", "taxonomy_id"], keep=False)]

In [89]:
duplicates.shape

(0, 12)